# Notebook 1 · Données (J1) · Étapes 1 et 2 : collecte et préparation

**Sujet 18 · Intentions d'achat sur un site marchand.** Repérer, pendant la visite, les sessions qui vont se terminer par un achat.

| Étape du cycle | Partie du notebook |
| --- | --- |
| 1 · Collecte | Partie 1 : charger les données officielles, contrôler leur qualité, explorer |
| 2 · Préparation | Partie 2 : nettoyer, séparer train / test **avant** de préparer, encoder et normaliser |

Exécutez les cellules dans l'ordre, de haut en bas.

In [ ]:
# Imports (toutes ces bibliothèques sont déjà installées dans Colab)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 0. Mise en place (identique dans les trois notebooks)

In [ ]:
# ===================== CONFIG : identique dans les 3 notebooks =====================
# Si vous modifiez une valeur ici, modifiez-la aussi dans les deux autres notebooks.
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"
LOCAL_PATH = "online_shoppers_intention.csv"   # utilisé si le fichier a été téléversé dans Colab

TARGET = "Revenue"
NUM_COLS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "SpecialDay",
]
LEAKY_COL = "PageValues"   # calculée à partir des transactions : testée avec et sans
CAT_COLS = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
BOOL_COLS = ["Weekend"]

FEATURE_SETS = {
    "sans_PageValues": NUM_COLS + CAT_COLS + BOOL_COLS,
    "avec_PageValues": NUM_COLS + [LEAKY_COL] + CAT_COLS + BOOL_COLS,
}

# Garde-fou : les 3 notebooks doivent obtenir exactement le même découpage.
EXPECTED_SPLIT = (9764, 2441)   # (lignes d'entraînement, lignes de test)

In [ ]:
# ===================== FONCTIONS COMMUNES : identiques dans les 3 notebooks =====================
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def load_raw():
    """Étape 1 : charge le CSV officiel (fichier local, sinon page UCI, sinon téléversement)."""
    if Path(LOCAL_PATH).exists():
        return pd.read_csv(LOCAL_PATH)
    try:
        return pd.read_csv(DATA_URL)
    except Exception as err:
        print("Téléchargement impossible :", err, "-> téléversez online_shoppers_intention.csv")
        from google.colab import files
        files.upload()
        return pd.read_csv(LOCAL_PATH)


def clean(df):
    """Nettoyage sans apprentissage (aucune statistique calculée) : sûr avant le découpage."""
    df = df.copy()
    df["Month"] = df["Month"].replace({"June": "Jun"})   # seul mois écrit en entier
    df["Weekend"] = df["Weekend"].astype(int)
    df[TARGET] = df[TARGET].astype(int)
    for col in CAT_COLS:                                 # les codes sont des catégories
        df[col] = df[col].astype(str)
    return df.drop_duplicates().reset_index(drop=True)   # avant le découpage, sinon fuite


def split(df):
    """Découpage stratifié, toujours avec la même graine."""
    X = df.drop(columns=TARGET)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    assert (len(X_train), len(X_test)) == EXPECTED_SPLIT, "Découpage différent des autres notebooks !"
    return X_train, X_test, y_train, y_test


def make_preprocessor(feature_set):
    """Étape 2 : à n'ajuster que sur l'entraînement (ou dans un Pipeline)."""
    num = [c for c in FEATURE_SETS[feature_set] if c in NUM_COLS + [LEAKY_COL]]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
        ("bool", "passthrough", BOOL_COLS),
    ], remainder="drop")   # remainder="drop" retire PageValues dans la variante « sans »


def build_estimator(name, params=None):
    """Étape 3 : les modèles candidats."""
    estimator = {
        "Reference_naive": DummyClassifier(strategy="most_frequent"),
        "Regression_logistique": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Foret_aleatoire": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient_boosting": HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE),
    }[name]
    if params:
        estimator.set_params(**params)
    return estimator


def make_pipeline(name, feature_set, params=None):
    return Pipeline([("prep", make_preprocessor(feature_set)),
                     ("model", build_estimator(name, params))])


CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Partie 1 · Collecte

### 1.1 Premier contact

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

### 1.2 Qualité : types, valeurs manquantes, doublons

In [ ]:
raw.info()
print("\nValeurs manquantes :", int(raw.isna().sum().sum()))
print("Lignes en double   :", int(raw.duplicated().sum()))
raw.describe().T

### 1.3 La cible : classes déséquilibrées

In [ ]:
print(raw[TARGET].value_counts())
print(raw[TARGET].value_counts(normalize=True).round(3))
sns.countplot(x=TARGET, data=raw)
plt.title("Répartition de Revenue (achat ou non)")
plt.show()

### 1.4 Les catégories : mois et type de visiteur

In [ ]:
print(raw["Month"].value_counts())       # remarque : 'June' en entier, pas de janvier ni d'avril
print(raw["VisitorType"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
sns.barplot(x="Month", y=TARGET, data=raw, order=order, ax=axes[0])
axes[0].set_title("Taux d'achat par mois")
sns.barplot(x="VisitorType", y=TARGET, data=raw, ax=axes[1])
axes[1].set_title("Taux d'achat par type de visiteur")
plt.tight_layout()
plt.show()

### 1.5 Le piège `PageValues`

In [ ]:
has_value = raw[LEAKY_COL] > 0
print("Taux d'achat quand PageValues > 0 :", round(raw.loc[has_value, TARGET].mean(), 3),
      "(", int(has_value.sum()), "sessions )")
print("Taux d'achat quand PageValues = 0 :", round(raw.loc[~has_value, TARGET].mean(), 3),
      "(", int((~has_value).sum()), "sessions )")

In [ ]:
num = NUM_COLS + [LEAKY_COL]
plt.figure(figsize=(9, 7))
sns.heatmap(raw[num + [TARGET]].astype(float).corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Corrélations des variables numériques avec la cible")
plt.show()

### 1.6 Constats (à rédiger en équipe)

- *Qualité :* …
- *Déséquilibre :* …
- *`PageValues` :* que sait-on vraiment pendant la visite ? …
- *Catégories codées en nombres :* …

# Partie 2 · Préparation

### 2.1 Nettoyage sans apprentissage

In [ ]:
df = clean(raw)
print("Lignes avant :", len(raw), "| après suppression des doublons :", len(df))
print(sorted(df["Month"].unique()))

### 2.2 Séparer **avant** de préparer

Le découpage est stratifié pour garder environ 15 % d'achats dans le train **et** dans le test.

In [ ]:
X_train, X_test, y_train, y_test = split(df)
print("Entraînement :", X_train.shape, "| Test :", X_test.shape)
print("Taux d'achat train :", round(y_train.mean(), 3), "| test :", round(y_test.mean(), 3))

### 2.3 Encoder et normaliser avec un `ColumnTransformer`

`fit` sur l'entraînement uniquement ; le test est seulement **transformé**.

In [ ]:
for name in FEATURE_SETS:
    prep = make_preprocessor(name)
    Xtr = prep.fit_transform(X_train)
    Xte = prep.transform(X_test)
    print(f"{name:16s} -> train {Xtr.shape} | test {Xte.shape}")

prep = make_preprocessor("sans_PageValues").fit(X_train)
print("\nExemples de colonnes créées :", list(prep.get_feature_names_out()[:12]))

### 2.4 Préparer une nouvelle session (comme le fera l'application)

In [ ]:
nouvelle_session = X_test.iloc[[0]]
display(nouvelle_session)
print(prep.transform(nouvelle_session).round(2))

## Bilan J1 (à rédiger)

- *Ce que nous avons appris sur les données :* …
- *Décisions de préparation et pourquoi :* …
- *Ce qui sera testé en J2 :* modèles avec et sans `PageValues`.